In [1]:
import pandas as pd
import numpy as np
import os

# ============================================================
# 0. Define File Paths
# ============================================================
# Using raw string r"" to handle Windows backslashes correctly
input_path = r"C:\Users\Surender\Downloads\marketing_campaign.csv"
output_path = r"C:\Users\Surender\Downloads\cleaned_marketing_campaign.csv"

print("Loading data from:", input_path)

# ============================================================
# 1. Load Data
# ============================================================
# This specific Kaggle dataset is tab-separated
df = pd.read_csv(input_path, sep='\t')
print("Raw shape:", df.shape)

# ============================================================
# 2. Standardize Column Names
# ============================================================
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

# ============================================================
# 3. Drop Constant Columns
# ============================================================
# Z_CostContact and Z_Revenue only contain a single value (3 and 11)
df = df.drop(columns=['z_costcontact', 'z_revenue'], errors='ignore')

# ============================================================
# 4. Handle Missing Values (Income)
# ============================================================
# Fill by Education group median, then fallback to global median
df['income'] = df['income'].fillna(df.groupby('education')['income'].transform('median'))
df['income'] = df['income'].fillna(df['income'].median())
print("Missing Income values filled.")

# ============================================================
# 5. Date Conversion (Dt_Customer)
# ============================================================
# Dataset uses DD-MM-YYYY format
df['dt_customer'] = pd.to_datetime(df['dt_customer'], dayfirst=True, errors='coerce')
df['customer_year'] = df['dt_customer'].dt.year
df['customer_month'] = df['dt_customer'].dt.month

# ============================================================
# 6. Fix Data Types & Remove Impossible Years
# ============================================================
df['year_birth'] = pd.to_numeric(df['year_birth'], errors='coerce')

# Remove rows with impossible birth years (1893, 1899, 1900)
before_rows = len(df)
df = df[df['year_birth'] >= 1900]
print(f"Removed {before_rows - len(df)} rows with impossible birth years.")

# Calculate Age (using max year in dataset, which is 2014)
df['age'] = 2014 - df['year_birth']

# ============================================================
# 7. Standardize Text Values
# ============================================================
df['education'] = df['education'].str.strip().str.title()
df['marital_status'] = df['marital_status'].str.strip().str.title()

# Merge sparse/inconsistent categories
df['marital_status'] = df['marital_status'].replace({
    'Alone': 'Single',
    'Absurd': 'Single',
    'Yolo': 'Single'
})

# ============================================================
# 8. Remove Duplicates
# ============================================================
before_dupes = len(df)
df = df.drop_duplicates(keep='first')
print(f"Removed {before_dupes - len(df)} duplicate rows.")

# ============================================================
# 9. Outlier Treatment (Income)
# ============================================================
# Calculate IQR for Income
Q1 = df['income'].quantile(0.25)
Q3 = df['income'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + (1.5 * IQR)

# Flag and cap instead of deleting to preserve rows
df['income_capped'] = np.where(df['income'] > upper_bound, upper_bound, df['income'])
print(f"Outlier threshold (Upper bound) for Income: {upper_bound:.2f}")

# ============================================================
# 10. Final Validation and Export
# ============================================================
print("\nCleaned shape:", df.shape)
print("Remaining missing values in key columns:")
print(df[['income', 'dt_customer']].isnull().sum())

assert df['income'].notna().all(), "Missing Income values found!"
assert pd.api.types.is_datetime64_any_dtype(df['dt_customer']), "Date not converted!"

df.to_csv(output_path, index=False)
print(f"\nSaved cleaned dataset to: {output_path}")

Loading data from: C:\Users\Surender\Downloads\marketing_campaign.csv
Raw shape: (2240, 29)
Missing Income values filled.
Removed 2 rows with impossible birth years.
Removed 0 duplicate rows.
Outlier threshold (Upper bound) for Income: 117405.75

Cleaned shape: (2238, 31)
Remaining missing values in key columns:
income         0
dt_customer    0
dtype: int64

Saved cleaned dataset to: C:\Users\Surender\Downloads\cleaned_marketing_campaign.csv
